### Lecture 3 -- Code Optimisation & Profiling

---

#### Milestone 1 -- Function-level profiling

In [16]:
import cProfile
import pstats
from mandelbrot_naive import compute_mandelbrot_naive
from mandelbrot_numpy import compute_mandelbrot_numpy

cProfile.run(
    "compute_mandelbrot_naive(100,1024,1024)",
    "naive_profile.prof"
)

cProfile.run(
    "compute_mandelbrot_numpy(100,1024,1024)",
    "numpy_profile.prof"
)

for name in ("naive_profile.prof", "numpy_profile.prof"):
    stats = pstats.Stats(name)
    stats.sort_stats("cumulative")
    stats.print_stats(10)

Tue Mar  3 17:25:34 2026    naive_profile.prof

         23067815 function calls in 19.834 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000   19.838   19.838 {built-in method builtins.exec}
        1    0.004    0.004   19.837   19.837 <string>:1(<module>)
        1    2.141    2.141   19.834   19.834 c:\Users\braec\Desktop\8.th semester\Numerical scientific computing\mandelbrot-nsc\MiniProject1\mandelbrot_naive.py:21(compute_mandelbrot_naive)
  1048576   13.946    0.000   17.557    0.000 c:\Users\braec\Desktop\8.th semester\Numerical scientific computing\mandelbrot-nsc\MiniProject1\mandelbrot_naive.py:11(mandelbrot_point)
 22018211    3.611    0.000    3.611    0.000 {built-in method builtins.abs}
     1024    0.132    0.000    0.132    0.000 c:\Users\braec\Desktop\8.th semester\Numerical scientific computing\mandelbrot-nsc\MiniProject1\mandelbrot_naive.py:23(<listcomp>)
        1    0.000    0

### Questions to Answer

---

#### Which function takes the most total time?

The **naive implementation** is slowest, with a runtime of **19.834 seconds**.  
most of the time is spent in `compute_mandelbrot_naive`, which is dominated by repeated calls to lower-level helper functions.

---

#### Are there functions called surprisingly many times?

- `mandelbrot_point` is called **1,048,576 times**, which is expected since it is executed once per pixel.
- However, the function `abs` is called **22,018,211 times**, which is surprisingly high.  
  Corresponding to an average of **approximately 21 calls per pixel**, making it a major contributor to the overall runtime.

---

#### How does the NumPy profile compare to the naive version?

The NumPy implementation performs only **377 function calls**, compared to **23,067,815** calls in the naive version.  
This drastic reduction in Python-level function calls results in a runtime of approximately **2 seconds**, yielding a **~10× speedup** over the naive implementation.

Profiling shows that the naive version is dominated by Python-level loops and repeated calls to `mandelbrot_point` and `abs`, whereas the NumPy version spends most of its time in optimized, vectorized C routines.

---

#### Milestone 2 -- Line-Level Profiling

In [17]:
%load_ext line_profiler


from mandelbrot_naive import compute_mandelbrot_naive
from mandelbrot_numpy import compute_mandelbrot_numpy


### Run only one at a time please

%lprun -f compute_mandelbrot_naive compute_mandelbrot_naive(100, 1024, 1024)


#%lprun -f compute_mandelbrot_numpy compute_mandelbrot_numpy(100, 1024, 1024)

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


Timer unit: 1e-07 s

Total time: 30.8651 s
File: c:\Users\braec\Desktop\8.th semester\Numerical scientific computing\mandelbrot-nsc\MiniProject1\mandelbrot_naive.py
Function: compute_mandelbrot_naive at line 21

Line #      Hits         Time  Per Hit   % Time  Line Contents
    21                                           def compute_mandelbrot_naive(max_iter, width, height, x_min = -2, x_max = 1, y_min = -1.5, y_max = 1.5):
    22                                           
    23         1    3469293.0 3.47e+06      1.1      mandelbrot_set = [[0 for _ in range(width)] for _ in range(height)]
    24                                           
    25      1025       7085.0      6.9      0.0      for j in range(height):
    26   1049600    3489148.0      3.3      1.1          for i in range(width):
    27   1048576    7926198.0      7.6      2.6              x = x_min + (x_max - x_min) * i / width
    28   1048576    6662022.0      6.4      2.2              y = y_min + (y_max - y_min) * j

### 1. cProfile on naive vs NumPy: How many functions appear in each profile? What does this difference tell you about where the work actually happens?


- **Naive implementation:** 7 functions appear.
- **NumPy implementation:** 45 functions appear, reduced to 10.

NumPy may list more distinct functions, but it executes very few Python-level calls. The heavy computation is performed inside optimized, compiled NumPy routines.
in tthe naive implementation, there is repeatedly executed a small number of Python functions inside inner loops, causing most of the runtime to be spent in the Python interpreter overhead.

### 2. line profiler on naive: Which lines dominate runtime? What fraction of total time is pent in the inner loop?

Line 30 which is the one executing the inner loop, the total fraction is 90.2% of total runtime



### 3. Why is NumPy faster than naive Python?

The computation is vectorized and executed in compiled C code

the Inner loops are eliminated


### 4. How could the naive version be made faster?

Based on the line profiler results:
- The inner loop is the main bottleneck.

How to make it faster
- Use NumPy vectorized operations 
- compile the inner loop.

The key takeaway: **eliminate Python work inside the inner loop**.

---

### Milestone 3 -- Implement & Compare Approaches

In [39]:
from numba import njit
import numpy as np



@njit
def mandelbrot_point(c, max_iter):
    z = 0j

    for i in range(max_iter):
        z = z*z + c
        if z.real*z.real + z.imag*z.imag <= 4.0:
            return i
    return max_iter


def compute_mandelbrot_hybrid(max_iter, width, height, x_min = -2, x_max = 1, y_min = -1.5, y_max = 1.5):

    mandelbrot_set = [[0 for _ in range(width)] for _ in range(height)]

    for j in range(height):
        for i in range(width):
            x = x_min + (x_max - x_min) * i / width
            y = y_min + (y_max - y_min) * j / height
            c = complex(x, y)
            mandelbrot_set[j][i] = mandelbrot_point(c, max_iter)
    return mandelbrot_set



@njit
def compute_mandelbrot_numba(max_iter, width, height, x_min=-2, x_max=1, y_min=-1.5, y_max=1.5):

    x = np.linspace(x_min, x_max, width)
    y = np.linspace(y_min, y_max, height)

    mandelbrot_set = np.zeros((height, width), dtype=np.int32)
    
    for j in range(height):
        for i in range(width):
            c = x[i] + 1j*y[j]
            z = 0j
            n= 0
            while n < max_iter and (z.real*z.real + z.imag*z.imag) <= 4.0:
                z = z*z + c; n += 1
            mandelbrot_set[j, i] = n

    return mandelbrot_set

In [ ]:
import time
import statistics


# Benchmark helper
def bench(fn, *args, runs=5):
    """Run fn(*args) multiple times and return the median execution time."""
    fn(*args)  # extra warm-up (triggers JIT compilation)
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn(*args)
        times.append(time.perf_counter() - t0)
    return statistics.median(times)


_ = compute_mandelbrot_hybrid(100, 64, 64)
_ = compute_mandelbrot_numba(100, 64, 64)

# Benchmark on large grid
t_hybrid = bench(compute_mandelbrot_hybrid, 100, 1024, 1024)
t_full = bench(compute_mandelbrot_numba, 100, 1024, 1024)

# Print results
print(f"Hybrid: {t_hybrid:.3f}s")
print(f"Fully compiled: {t_full:.3f}s")
print(f"Ratio: {t_hybrid / t_full:.1f}x")

Hybrid: 0.483s
Fully compiled: 0.055s
Ratio: 8.7x


---


### Milestone 3 -- all versions comparison

 Implementation | Median Time (s) |  Speedup vs Naive |
|----------------|----------------|-------------------|
| Naive          | 3.4218         | -                 |
| NumPy          | 0.9988         | 3.43              |
| Numba          | 0.055          | 62.21             |

---

### Milestone 4 --- Test Different Precisions